# 6.9 Lion, LAMB, and Large Batches

Large-batch optimizers control direction and layerwise scale so huge updates do not trample useful geometry.

The local direction is not enough when batches become large. Lion uses a sign direction, while LAMB rescales an update by the layer's parameter norm. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def split_scale(X, y):
    if len(y) > 300:
        x_small, _, y_small, _ = train_test_split(X, y, train_size=300, random_state=6, stratify=y)
    else:
        x_small = X
        y_small = y
    x_tr, x_te, y_tr, y_te = train_test_split(x_small, y_small, test_size=0.4, random_state=0, stratify=y_small)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / np.sum(ez, axis=1, keepdims=True)


def relu(z):
    return np.maximum(z, 0.0)


def init_weights(n_in, n_hidden, n_out, mode, seed):
    rng = np.random.default_rng(seed)
    if mode == "xavier":
        scale1 = math.sqrt(2.0 / (n_in + n_hidden))
        scale2 = math.sqrt(2.0 / (n_hidden + n_out))
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "he":
        scale1 = math.sqrt(2.0 / n_in)
        scale2 = math.sqrt(2.0 / n_hidden)
        W1 = rng.normal(0.0, scale1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, scale2, size=(n_hidden, n_out))
    elif mode == "tiny":
        W1 = rng.normal(0.0, 0.01, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.01, size=(n_hidden, n_out))
    elif mode == "large":
        W1 = rng.normal(0.0, 2.0, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 2.0, size=(n_hidden, n_out))
    elif mode == "orthogonal":
        Q1, _ = np.linalg.qr(rng.normal(size=(n_in, max(n_in, n_hidden))))
        Q2, _ = np.linalg.qr(rng.normal(size=(n_hidden, max(n_hidden, n_out))))
        W1 = Q1[:, :n_hidden]
        W2 = Q2[:, :n_out]
    else:
        W1 = rng.normal(0.0, 0.1, size=(n_in, n_hidden))
        W2 = rng.normal(0.0, 0.1, size=(n_hidden, n_out))
    b1 = np.zeros(n_hidden)
    b2 = np.zeros(n_out)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}


def forward(params, X, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    W1 = params["W1"]
    if dropconnect_p > 0.0 and rng is not None:
        keep_w = 1.0 - dropconnect_p
        mask_w = rng.binomial(1, keep_w, size=W1.shape) / keep_w
        W1 = W1 * mask_w
    z1 = X @ W1 + params["b1"]
    h1 = relu(z1)
    mask = None
    if dropout_p > 0.0 and rng is not None:
        keep = 1.0 - dropout_p
        mask = rng.binomial(1, keep, size=h1.shape) / keep
        h1 = h1 * mask
    logits = h1 @ params["W2"] + params["b2"]
    return z1, h1, logits, mask


def loss_and_grads(params, X, y, dropout_p=0.0, rng=None, dropconnect_p=0.0):
    classes = params["b2"].shape[0]
    z1, h1, logits, mask = forward(params, X, dropout_p, rng, dropconnect_p)
    probs = softmax(logits)
    target = one_hot(y, classes)
    loss = -np.mean(np.sum(target * np.log(probs + 1e-12), axis=1))
    dlogits = (probs - target) / len(y)
    dW2 = h1.T @ dlogits
    db2 = np.sum(dlogits, axis=0)
    dh1 = dlogits @ params["W2"].T
    if mask is not None:
        dh1 = dh1 * mask
    dz1 = dh1 * (z1 > 0.0)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0)
    grads = {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
    return loss, grads


def predict(params, X):
    _, _, logits, _ = forward(params, X)
    return np.argmax(logits, axis=1)


def eval_loss(params, X, y):
    classes = params["b2"].shape[0]
    _, _, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    return -float(np.mean(np.sum(target * np.log(probs + 1e-12), axis=1)))


def vector_norm(params):
    total = 0.0
    for value in params.values():
        total += float(np.sum(value * value))
    return math.sqrt(total)


def kfac_precondition_grads(params, X, y, grads, damping):
    classes = params["b2"].shape[0]
    z1, h1, logits, _ = forward(params, X)
    probs = softmax(logits)
    target = one_hot(y, classes)
    dlogits = (probs - target) / len(y)
    dh1 = dlogits @ params["W2"].T
    dz1 = dh1 * (z1 > 0.0)
    out = {key: value.copy() for key, value in grads.items()}
    A1 = X.T @ X / len(y) + damping * np.eye(X.shape[1])
    S1 = dz1.T @ dz1 / len(y) + damping * np.eye(dz1.shape[1])
    A2 = h1.T @ h1 / len(y) + damping * np.eye(h1.shape[1])
    S2 = dlogits.T @ dlogits / len(y) + damping * np.eye(dlogits.shape[1])
    out["W1"] = np.linalg.solve(A1, grads["W1"]) @ np.linalg.inv(S1)
    out["W2"] = np.linalg.solve(A2, grads["W2"]) @ np.linalg.inv(S2)
    return out


def apply_update(params, grads, state, method, lr, t, config):
    beta1 = config.get("beta1", 0.9)
    beta2 = config.get("beta2", 0.999)
    eps = config.get("eps", 1e-8)
    mu = config.get("momentum", 0.0)
    weight_decay = config.get("weight_decay", 0.0)
    for key in params:
        grad = grads[key]
        if method == "sgd":
            update = -lr * grad
        elif method == "momentum":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        elif method == "adagrad":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc += grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "rmsprop":
            acc = state.setdefault("acc_" + key, np.zeros_like(params[key]))
            acc *= beta2
            acc += (1.0 - beta2) * grad * grad
            update = -lr * grad / (np.sqrt(acc) + eps)
        elif method == "adam" or method == "adamw":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            m_hat = m / (1.0 - beta1 ** t)
            v_hat = v / (1.0 - beta2 ** t)
            update = -lr * m_hat / (np.sqrt(v_hat) + eps)
            if method == "adamw" and key.startswith("W"):
                update -= lr * weight_decay * params[key]
        elif method == "lion":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            blended = beta1 * m + (1.0 - beta1) * grad
            update = -lr * np.sign(blended)
            m *= beta2
            m += (1.0 - beta2) * grad
        elif method == "lamb":
            m = state.setdefault("m_" + key, np.zeros_like(params[key]))
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            m *= beta1
            m += (1.0 - beta1) * grad
            v *= beta2
            v += (1.0 - beta2) * grad * grad
            raw = m / (np.sqrt(v) + eps)
            if key.startswith("W"):
                raw += weight_decay * params[key]
            ratio = np.linalg.norm(params[key]) / (np.linalg.norm(raw) + eps)
            ratio = float(np.clip(ratio, 0.1, 10.0))
            update = -lr * ratio * raw
        elif method == "nesterov":
            v = state.setdefault("v_" + key, np.zeros_like(params[key]))
            v *= mu
            v -= lr * grad
            update = v
        else:
            update = -lr * grad
        if weight_decay > 0.0 and method not in ["adamw", "lamb"] and key.startswith("W"):
            update -= lr * weight_decay * params[key]
        params[key] += update


def train_mlp(x_tr, y_tr, x_te, y_te, method="sgd", init="he", epochs=12, lr=0.05, hidden=8, batch_size=None, config=None, dropout_p=0.0, dropconnect_p=0.0, seed=0, early_patience=None):
    if config is None:
        config = {}
    classes = int(np.max(y_tr)) + 1
    params = init_weights(x_tr.shape[1], hidden, classes, init, seed)
    state = {}
    rng = np.random.default_rng(seed + 100)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "norm": []}
    best_loss = float("inf")
    best_params = None
    bad_epochs = 0
    n = len(y_tr)
    if batch_size is None:
        batch_size = n
    for epoch in range(1, epochs + 1):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            if method == "nesterov":
                lookahead = {}
                for key in params:
                    velocity = state.setdefault("v_" + key, np.zeros_like(params[key]))
                    lookahead[key] = params[key].copy()
                    params[key] += config.get("momentum", 0.9) * velocity
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
                for key in params:
                    params[key] = lookahead[key]
            else:
                loss, grads = loss_and_grads(params, x_tr[idx], y_tr[idx], dropout_p, rng, dropconnect_p)
            if method == "kfac":
                grads = kfac_precondition_grads(params, x_tr[idx], y_tr[idx], grads, config.get("damping", 0.03))
                apply_update(params, grads, state, "sgd", lr, epoch, config)
            else:
                apply_update(params, grads, state, method, lr, epoch, config)
        train_loss = eval_loss(params, x_tr, y_tr)
        val_loss = eval_loss(params, x_te, y_te)
        train_acc = accuracy_score(y_tr, predict(params, x_tr))
        val_acc = accuracy_score(y_te, predict(params, x_te))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["norm"].append(vector_norm(params))
        if val_loss < best_loss:
            best_loss = val_loss
            best_params = {key: value.copy() for key, value in params.items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if early_patience is not None and bad_epochs >= early_patience:
            params = best_params
            break
    return params, history


def run_component_ladder(variants, metric="accuracy", epochs=12, hidden=16):
    rows = []
    histories = {}
    artifacts = {}
    for rung_index, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        histories[name] = {}
        artifacts[name] = {}
        for variant in variants:
            params, hist = train_mlp(
                x_tr,
                y_tr,
                x_te,
                y_te,
                method=variant.get("method", "sgd"),
                init=variant.get("init", "he"),
                epochs=variant.get("epochs", epochs),
                lr=variant.get("lr", 0.05),
                hidden=hidden,
                batch_size=variant.get("batch_size"),
                config=variant.get("config", {}),
                dropout_p=variant.get("dropout_p", 0.0),
                dropconnect_p=variant.get("dropconnect_p", 0.0),
                seed=variant.get("seed", 10 + rung_index),
                early_patience=variant.get("early_patience"),
            )
            preds = predict(params, x_te)
            acc = accuracy_score(y_te, preds)
            val_loss = eval_loss(params, x_te, y_te)
            value = acc if metric == "accuracy" else val_loss
            rows.append({"rung": name, "variant": variant["name"], "accuracy": acc, "loss": val_loss, "metric": value})
            histories[name][variant["name"]] = hist
            artifacts[name][variant["name"]] = (x_te, y_te, preds)
    return rows, histories, artifacts


def print_table(rows, metric_name):
    print(f"{'rung':34s} {'variant':18s} {metric_name:>10s} {'acc':>8s} {'loss':>8s}")
    for row in rows:
        print(f"{row['rung'][:34]:34s} {row['variant'][:18]:18s} {row['metric']:10.3f} {row['accuracy']:8.3f} {row['loss']:8.3f}")


def plot_results(rows, histories, artifacts, metric_name, best_variant):
    rung_names = list(histories.keys())
    fig, axes = plt.subplots(2, len(rung_names), figsize=(3.2 * len(rung_names), 6.4))
    for col, rung in enumerate(rung_names):
        x_te, y_te, preds = artifacts[rung][best_variant]
        if x_te.shape[1] > 2:
            shown = PCA(n_components=2, random_state=0).fit_transform(x_te)
        else:
            shown = x_te[:, :2]
        axes[0, col].scatter(shown[:, 0], shown[:, 1], c=preds, s=12, cmap="tab10", alpha=0.85)
        axes[0, col].set_title(rung.split("(")[0].strip())
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])
        hist = histories[rung][best_variant]
        curve_key = "val_acc" if metric_name == "accuracy" else "val_loss"
        axes[1, col].plot(hist[curve_key], label=best_variant)
        axes[1, col].set_xlabel("epoch")
        axes[1, col].set_title(metric_name)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 3.5))
    variants = sorted({row["variant"] for row in rows})
    for variant in variants:
        vals = [row["metric"] for row in rows if row["variant"] == variant]
        ax.plot(range(1, len(vals) + 1), vals, marker="o", label=variant)
    ax.set_xticks(range(1, len(rung_names) + 1))
    ax.set_xticklabels([f"D{i}" for i in range(1, len(rung_names) + 1)])
    ax.set_ylabel(metric_name)
    ax.set_title("Same ladder, component varied")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## The concept, built once: sign and trust ratio

The lesson formula is
$$\theta_t=\theta_{t-1}-\eta\,\mathrm{sign}(m_t),\qquad r=\frac{\|\theta\|}{\|u\|}.$$
The lesson's local SGD number uses $\eta=0.090$, $g=1.500$, and $\theta=2.000$, giving $1.865$.

In [ ]:

def large_batch_optimizer_sweep():
    theta = 2.0
    eta = 0.090
    gradient = 1.500
    m = 0.9 * 0.0 + 0.1 * gradient
    lion_theta = theta - eta * np.sign(m)
    plain_theta = theta - eta * gradient
    return plain_theta, lion_theta, m

plain_theta, lion_theta, m = large_batch_optimizer_sweep()
print(plain_theta, lion_theta, m)
assert abs(plain_theta - 1.865) < 1e-12
assert abs(lion_theta - 1.91) < 1e-12
assert abs(m - 0.15) < 1e-12


LAMB computes a raw adaptive update and multiplies it by a trust ratio. This preserves the direction but matches update scale to layer scale.

In [ ]:

theta = np.array([3.0, 4.0])
raw_update = np.array([0.6, 0.8])
trust_ratio = np.linalg.norm(theta) / np.linalg.norm(raw_update)
trusted = 0.09 * trust_ratio * raw_update
print("trust ratio", trust_ratio, "trusted update norm", np.linalg.norm(trusted))
assert abs(trust_ratio - 5.0) < 1e-12
assert abs(np.linalg.norm(trusted) - 0.45) < 1e-12


## The dataset ladder

Every topic uses the same `clf_digits_ladder()` and the same small MLP. Only the named optimizer or regularization component changes from variant to variant.

In [ ]:

rungs = clf_digits_ladder()
for name, X, y in rungs:
    classes = np.unique(y)
    print(f"{name:38s} shape={X.shape} classes={len(classes)} sample_y={y[:8].tolist()}")
print("D1 sample X:")
print(rungs[0][1])


## Run the same method across D1-D5

The architecture, splits, scaling, and seed policy stay fixed. The table reports one comparable metric per rung.

In [ ]:

variants = [
    {"name": "Lion small-batch", "method": "lion", "lr": 0.003, "batch_size": 32},
    {"name": "LAMB large-batch", "method": "lamb", "lr": 0.01, "batch_size": 256, "config": {"weight_decay": 0.01}},
    {"name": "Adam large-batch", "method": "adam", "lr": 0.01, "batch_size": 256},
]

rows, histories, artifacts = run_component_ladder(variants, metric="accuracy", epochs=10, hidden=8)
print_table(rows, "accuracy")


## Results visualization

Top row: small multiples of held-out predictions. Bottom row: validation curves for the highlighted variant, followed by the component summary curve.

In [ ]:

plot_results(rows, histories, artifacts, "accuracy", "LAMB large-batch")


## Pitfall on D5: smooth large-batch curves can generalize worse

A large batch often gives less noisy training, but that is not the same as best validation accuracy. Learning-rate scaling and warmup-style smaller initial steps are the fix.

In [ ]:

name, X, y = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_scale(X, y)
_, too_big = train_mlp(x_tr, y_tr, x_te, y_te, method="adam", lr=0.04, batch_size=512, epochs=10, seed=99)
_, fixed = train_mlp(x_tr, y_tr, x_te, y_te, method="lamb", lr=0.01, batch_size=256, epochs=10, config={"weight_decay": 0.01}, seed=99)
print("too-large batch/lr val acc", round(too_big["val_acc"][-1], 3))
print("trust-ratio fixed val acc", round(fixed["val_acc"][-1], 3))
assert fixed["val_acc"][-1] >= too_big["val_acc"][-1] - 0.15


## Evaluate it + Practice

- Main metric: held-out accuracy on every D1-D5 rung, compared with a no-skill baseline near random guessing.
- Sanity check: D1 XOR should improve above chance once the hidden ReLU layer is active.
- Ablation: turn off the LAMB trust ratio by switching to Adam at the same large batch; the metric should drop or the curve should become less stable.
- Failure signals: exploding loss, flat accuracy near chance, or a D5 train/validation gap that moves in opposite directions.
- Reproducibility: seeds are fixed and the ladder uses sklearn-bundled data only.

Practice 1: Change one hyperparameter in the strongest variant and rerun the summary curve.

Practice 2: Add a new diagnostic printout that distinguishes train accuracy from validation accuracy.

Practice 3: Explain why D5 is harder than D1 using the table and one plotted curve.